# Loading Libraries and Packages

In [2]:
import copy, math, os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.transforms.functional import affine
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from collections import defaultdict

# Setting the random seed for reproducibility
np.random.seed(42)

import warnings
warnings.filterwarnings("ignore")

if torch.backends.mps.is_available():
    print(f"MPS available. \nUsing MPS as runtime.")
    device = torch.device("mps")
    
elif torch.cuda.is_available():
    print(f"CUDA available. \nUsing CUDA as runtime.")
    device = torch.device("cuda")

else:
    print(f"No GPU available. \nUsing CPU runtime.")
    device = torch.device('cpu')

MPS available. 
Using MPS as runtime.


# 2.4 CNN from Scratch & Receptive Fields

## Receptive Field Calculation

In [2]:
# Receptive Field Calculator (Conv + Pool)
def compute_receptive_field(config):
    r, j = 1, 1

    print("\nReceptive Field Calculation:")
    print(f"{'Layer':<10} {'Type':<6} {'k':<5} {'s':<5} {'r':<10} {'j':<10}")

    layer_sequence = [
        ("conv1", "conv"),
        ("pool1", "pool"),
        ("conv2", "conv"),
        ("pool2", "pool"),
        ("conv3", "conv"),
        ("pool3", "pool"),
    ]

    for name, ltype in layer_sequence:
        k = config[name]["kernel_size"]
        s = config[name]["stride"]

        r = r + (k - 1) * j
        j = j * s

        print(f"{name:<10} {ltype:<6} {k:<5} {s:<5} {r:<10} {j:<10}")

    print(f"\nFinal Receptive Field: {r} x {r}\n")

## CNN Model Class

In [3]:
# CNN Model
class DemoCNN(nn.Module):
    def __init__(self, config):
        super(DemoCNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv2d(**config["conv1"])
        self.bn1 = nn.BatchNorm2d(config["conv1"]["out_channels"])
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(**config["pool1"])

        # Block 2
        self.conv2 = nn.Conv2d(**config["conv2"])
        self.bn2 = nn.BatchNorm2d(config["conv2"]["out_channels"])
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(**config["pool2"])

        # Block 3
        self.conv3 = nn.Conv2d(**config["conv3"])
        self.bn3 = nn.BatchNorm2d(config["conv3"]["out_channels"])
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(**config["pool3"])

    def forward(self, x):
        print("\n================ INPUT ================")
        print(f"Input: {x.shape}")

        # ---------------- BLOCK 1 ----------------
        print("\n--- Block 1 ---")
        x = self.conv1(x)
        print(f"  Conv1   → {x.shape}")
        x = self.bn1(x)
        print(f"  BN1     → {x.shape}")
        x = self.relu1(x)
        print(f"  ReLU1   → {x.shape}")
        x = self.pool1(x)
        print(f"  Pool1   → {x.shape}")

        # ---------------- BLOCK 2 ----------------
        print("\n--- Block 2 ---")
        x = self.conv2(x)
        print(f"  Conv2   → {x.shape}")
        x = self.bn2(x)
        print(f"  BN2     → {x.shape}")
        x = self.relu2(x)
        print(f"  ReLU2   → {x.shape}")
        x = self.pool2(x)
        print(f"  Pool2   → {x.shape}")

        # ---------------- BLOCK 3 ----------------
        print("\n--- Block 3 ---")
        x = self.conv3(x)
        print(f"  Conv3   → {x.shape}")
        x = self.bn3(x)
        print(f"  BN3     → {x.shape}")
        x = self.relu3(x)
        print(f"  ReLU3   → {x.shape}")
        x = self.pool3(x)
        print(f"  Pool3   → {x.shape}")
        print("\n============= FINAL OUTPUT =============")
        print(f"Output: {x.shape}\n")
        return x

## Driver Code

In [4]:
config = {
    "conv1": {
        "in_channels": 1,
        "out_channels": 16,
        "kernel_size": 3,
        "stride": 1,
        "padding": 1
    },
    "pool1": {"kernel_size": 2, "stride": 2},

    "conv2": {
        "in_channels": 16,
        "out_channels": 32,
        "kernel_size": 3,
        "stride": 1,
        "padding": 1
    },
    "pool2": {"kernel_size": 2, "stride": 2},

    "conv3": {
        "in_channels": 32,
        "out_channels": 64,
        "kernel_size": 3,
        "stride": 1,
        "padding": 1
    },
    "pool3": {"kernel_size": 2, "stride": 2}
}


model = DemoCNN(config) # Model
x = torch.randn(1, 1, 128, 128) # Sample Input
output = model(x) # Forward pass


================ INPUT ================
Input: torch.Size([1, 1, 128, 128])

--- Block 1 ---
  Conv1   → torch.Size([1, 16, 128, 128])
  BN1     → torch.Size([1, 16, 128, 128])
  ReLU1   → torch.Size([1, 16, 128, 128])
  Pool1   → torch.Size([1, 16, 64, 64])

--- Block 2 ---
  Conv2   → torch.Size([1, 32, 64, 64])
  BN2     → torch.Size([1, 32, 64, 64])
  ReLU2   → torch.Size([1, 32, 64, 64])
  Pool2   → torch.Size([1, 32, 32, 32])

--- Block 3 ---
  Conv3   → torch.Size([1, 64, 32, 32])
  BN3     → torch.Size([1, 64, 32, 32])
  ReLU3   → torch.Size([1, 64, 32, 32])
  Pool3   → torch.Size([1, 64, 16, 16])

============= FINAL OUTPUT =============
Output: torch.Size([1, 64, 16, 16])



In [5]:
# Receptive Field
compute_receptive_field(config)


Receptive Field Calculation:
Layer      Type   k     s     r          j         
conv1      conv   3     1     3          1         
pool1      pool   2     2     4          2         
conv2      conv   3     1     8          2         
pool2      pool   2     2     10         4         
conv3      conv   3     1     18         4         
pool3      pool   2     2     22         8         

Final Receptive Field: 22 x 22



# 2.5 CNN Regression Head

## Healthy and Diseased Criteria (For Bounding Box calculation)

In [28]:
def compute_healthy_hsv_stats(segmented_root):
    """
    Computes per-crop HSV mean and std from healthy segmented images.
    
    Returns:
        stats_dict: {
            crop_name: {
                'H_mean', 'H_std',
                'S_mean', 'S_std'
            }
        }
    """
    stats = defaultdict(list)

    for class_name in os.listdir(segmented_root):
        if "healthy" not in class_name:
            continue

        class_path = os.path.join(segmented_root, class_name)
        if not os.path.isdir(class_path):
            continue

        crop = class_name.split("___")[0]

        for file in os.listdir(class_path):
            path = os.path.join(class_path, file)

            if not os.path.isfile(path):
                continue

            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Leaf mask (non-black)
            leaf_mask = np.any(img > 10, axis=2)

            hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

            H = hsv[..., 0][leaf_mask]
            S = hsv[..., 1][leaf_mask]

            stats[crop].append((H, S))

    # Aggregate stats
    stats_dict = {}

    for crop, values in stats.items():
        H_all = np.concatenate([v[0] for v in values])
        S_all = np.concatenate([v[1] for v in values])

        stats_dict[crop] = {
            "H_mean": H_all.mean(),
            "H_std": H_all.std(),
            "S_mean": S_all.mean(),
            "S_std": S_all.std(),
        }

    return stats_dict

## Dataset and Data Loader

In [47]:
#########  DATASET CLASS  #########
class PlantVillageDataset(Dataset):
    def __init__(self, root_dir, segmented_stats, transform=None):
        """
        Args:
            root_dir: original image folder
            segmented_stats: precomputed stats dict (per crop)
            transform: torchvision transforms
        """
        self.root_dir = root_dir
        self.transform = transform
        self.segmented_stats = segmented_stats

        self.samples = []
        self.labels = []   # optional (kept for compatibility)

        self.valid_extensions = (".jpg", ".jpeg", ".png", ".JPG")

        class_names = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d)) ])

        for class_name in class_names:
            # Exclude healthy classes
            if "healthy" in class_name:
                continue

            class_dir = os.path.join(root_dir, class_name)

            for file_name in os.listdir(class_dir):
                if not file_name.lower().endswith(self.valid_extensions):
                    continue

                path = os.path.join(class_dir, file_name)

                if not os.path.isfile(path):
                    continue

                self.samples.append((path, class_name))
                self.labels.append(class_name)

        if len(self.samples) == 0:
            raise RuntimeError("No valid samples found.")

    def __len__(self):
        return len(self.samples)

    def compute_severity(self, image_np, crop):
        """
        Compute severity using:
        - HSV deviation from healthy stats (if available)
        - Otherwise fallback to generic HSV rule
        - Bounding box ratio
        """

        hsv = cv2.cvtColor(image_np, cv2.COLOR_RGB2HSV)
    
        H = hsv[..., 0]
        S = hsv[..., 1]
    
        # Leaf mask (non-black)
        leaf_mask = np.any(image_np > 10, axis=2)
    
        # -------------------------------
        # CASE 1: Stats available
        # -------------------------------
        if crop in self.segmented_stats:
    
            stats = self.segmented_stats[crop]
    
            k = 2
            H_low  = stats["H_mean"] - k * stats["H_std"]
            H_high = stats["H_mean"] + k * stats["H_std"]
            S_low  = stats["S_mean"] - k * stats["S_std"]
    
            healthy_mask = (
                (H >= H_low) & (H <= H_high) &
                (S >= S_low)
            )
    
        # -------------------------------
        # CASE 2: Fallback (no stats)
        # -------------------------------
        else:
            # Generic green detection (HSV)
            healthy_mask = (
                (H >= 35) & (H <= 85) &   # green hue
                (S >= 40)                 # reasonably saturated
            )
    
        # Diseased mask
        diseased_mask = leaf_mask & (~healthy_mask)
    
        # Edge case: no leaf pixels
        if leaf_mask.sum() == 0:
            return 0.0
    
        # Edge case: no diseased pixels
        if diseased_mask.sum() == 0:
            return 0.0
    
        # --- Diseased bounding box ---
        y_d, x_d = np.where(diseased_mask)
        d_xmin, d_xmax = x_d.min(), x_d.max()
        d_ymin, d_ymax = y_d.min(), y_d.max()
    
        diseased_area = (d_xmax - d_xmin + 1) * (d_ymax - d_ymin + 1)
    
        # --- Leaf bounding box ---
        y_l, x_l = np.where(leaf_mask)
        l_xmin, l_xmax = x_l.min(), x_l.max()
        l_ymin, l_ymax = y_l.min(), y_l.max()
    
        leaf_area = (l_xmax - l_xmin + 1) * (l_ymax - l_ymin + 1)
    
        severity = diseased_area / leaf_area
    
        return float(severity)
    

    def __getitem__(self, idx):
        path, class_name = self.samples[idx]

        # Load image (PIL → numpy)
        image = Image.open(path).convert("RGB")
        image_np = np.array(image)

        # Extract crop name
        crop = class_name.split("___")[0]

        # Compute severity
        severity = self.compute_severity(image_np, crop)

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(severity, dtype=torch.float32)

In [3]:
#########  STRATIFIED SAMPLING  #########

def stratified_split(dataset, random_state=42):
    indices = list(range(len(dataset)))
    labels = dataset.labels

    # Step 1: Train (70) vs Temp (30)
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.3,
        stratify=labels,
        random_state=random_state
    )

    # Labels for temp split
    temp_labels = [labels[i] for i in temp_idx]

    # Step 2: Temp → Val (15) + Test (15)
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.5,
        stratify=temp_labels,
        random_state=random_state
    )

    return train_idx, val_idx, test_idx

In [38]:
#########  DATALOADER CREATION  #########

def create_dataloaders(root_dir,segmented_stats, transform = None, train_batch_size=512, test_batch_size = 256):    
    dataset = PlantVillageDataset(root_dir, segmented_stats, transform=transform)
    train_idx, val_idx, test_idx = stratified_split(dataset)

    train_dataset = Subset(dataset, train_idx)
    val_dataset   = Subset(dataset, val_idx)
    test_dataset  = Subset(dataset, test_idx)

    train_loader = DataLoader(
        train_dataset,
        batch_size=train_batch_size,
        shuffle=True
        # num_workers=2,
        # pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=test_batch_size,
        shuffle=False
        # num_workers=2,
        # pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=test_batch_size,
        shuffle=False
        # num_workers=2,
        # pin_memory=True
    )

    return train_loader, val_loader, test_loader

## CNN Architecture

In [12]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x):
        return self.block(x)

In [13]:
class CNNRegressor(nn.Module):
    def __init__(self):
        super().__init__()

        # Feature extractor
        self.features = nn.Sequential(
            ConvBlock(3, 32),    # 224 → 112
            ConvBlock(32, 64),   # 112 → 56
            ConvBlock(64, 128),  # 56 → 28
            ConvBlock(128, 256), # 28 → 14
        )

        # Adaptive pooling → fixed size
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Regression head
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)   # scalar output
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.regressor(x)
        
        return x.squeeze(1)  # shape: (batch,)

## Utility Function

In [33]:
def compute_mae(preds, targets):
    return torch.mean(torch.abs(preds - targets)).item()

In [34]:
def evaluate(model, loader, device):
    model.eval()
    mae = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)
            mae += torch.mean(torch.abs(preds - targets)).item()

    return mae / len(loader)

## Training Loop

In [41]:
def train_model( model, train_loader, val_loader,
    optimizer, criterion, device,
    num_epochs=20,
    patience=5,
    verbose=True
):
    model.to(device)

    best_val_loss = float("inf")
    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_mae": []
    }

    for epoch in range(num_epochs):
        # ================= TRAIN =================
        model.train()
        running_loss = 0

        for batch_idx, (images, targets) in enumerate(train_loader):
            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)
            loss = criterion(preds, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # -------- Verbose Batch Logging --------
            if verbose and batch_idx % 50 == 0:
                print(f"[Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)}] "
                      f"Loss: {loss.item():.4f}")

        train_loss = running_loss / len(train_loader)

        # ================= VALIDATION =================
        model.eval()
        val_loss = 0
        val_mae = 0

        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device)
                targets = targets.to(device)

                preds = model(images)
                loss = criterion(preds, targets)

                val_loss += loss.item()
                val_mae += torch.mean(torch.abs(preds - targets)).item()

        val_loss /= len(val_loader)
        val_mae /= len(val_loader)

        # ================= LOG =================
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_mae"].append(val_mae)

        print(f"\nEpoch [{epoch+1}/{num_epochs}] Summary:")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss:   {val_loss:.4f}")
        print(f"Val MAE:    {val_mae:.4f}\n")

        # ================= EARLY STOPPING =================
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pth")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    return history

## Driver Code

In [48]:
save_dir = "../Models"
model_num = 2
save_path = os.path.join(save_dir, f"2_5_{model_num}.pth")

# Create directory if it does not exist
os.makedirs(save_dir, exist_ok=True)

# ================= DATA =================
transform = transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    ])

print("Creating Bounding Box Stats...")
segmented_folder = "../plantvillage dataset/segmented"
segmented_stats = compute_healthy_hsv_stats(segmented_folder)

print("Creating Data Loaders...")
train_loader, val_loader, test_loader = create_dataloaders(
    root_dir= segmented_folder,
    segmented_stats = segmented_stats, transform = transform,
    train_batch_size= 64, test_batch_size = 32)

print("Done.")

Creating Bounding Box Stats...
Creating Data Loaders...
Done.


In [49]:
# ================= MODEL =================
model = CNNRegressor()

# ================= LOSS =================
criterion = nn.MSELoss()

# ================= OPTIMIZER =================
optimizer = optim.Adam(model.parameters(), lr=5e-4)

# ================= TRAIN =================
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=20,
    patience=3,
    verbose=True
)

# ================= SAVE FINAL MODEL =================
torch.save(model.state_dict(), save_path)
print(f"Model saved to: {save_path}")

[Epoch 1 | Batch 0/429] Loss: 0.9132
[Epoch 1 | Batch 50/429] Loss: 0.0020
[Epoch 1 | Batch 100/429] Loss: 0.0011
[Epoch 1 | Batch 150/429] Loss: 0.0012
[Epoch 1 | Batch 200/429] Loss: 0.0009
[Epoch 1 | Batch 250/429] Loss: 0.0006
[Epoch 1 | Batch 300/429] Loss: 0.0006
[Epoch 1 | Batch 350/429] Loss: 0.0007
[Epoch 1 | Batch 400/429] Loss: 0.0003

Epoch [1/20] Summary:
Train Loss: 0.0046
Val Loss:   0.0009
Val MAE:    0.0244

[Epoch 2 | Batch 0/429] Loss: 0.0002
[Epoch 2 | Batch 50/429] Loss: 0.0015
[Epoch 2 | Batch 100/429] Loss: 0.0006
[Epoch 2 | Batch 150/429] Loss: 0.0010
[Epoch 2 | Batch 200/429] Loss: 0.0013
[Epoch 2 | Batch 250/429] Loss: 0.0004
[Epoch 2 | Batch 300/429] Loss: 0.0004
[Epoch 2 | Batch 350/429] Loss: 0.0002
[Epoch 2 | Batch 400/429] Loss: 0.0031

Epoch [2/20] Summary:
Train Loss: 0.0007
Val Loss:   0.0006
Val MAE:    0.0131

[Epoch 3 | Batch 0/429] Loss: 0.0025
[Epoch 3 | Batch 50/429] Loss: 0.0006
[Epoch 3 | Batch 100/429] Loss: 0.0004
[Epoch 3 | Batch 150/429] Lo

In [51]:
train_mae = evaluate(model, train_loader, device)
test_mae = evaluate(model, test_loader, device)
val_mae = evaluate(model, val_loader, device)

print(f"Train MAE: {train_mae:.6f}")
print(f"Val MAE: {val_mae:.6f}")
print(f"Test MAE: {test_mae:.6f}")

Train MAE: 0.015586
Val MAE: 0.015629
Test MAE: 0.015626


# 2.6 Translation Invariance

## Dataset Class

In [4]:
#########  DATASET CLASS  #########

class PlantVillageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.labels_to_index = {}
        self.index_to_labels = {}
        self.samples = []
        self.labels = []   # needed for stratified split

        self.valid_extensions = (".jpg",".jpeg", ".png")
        
        # List of class names (folders in the root directory)
        class_names = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))])

        for labels_index,class_name in enumerate(class_names):
            class_dir = os.path.join(root_dir, class_name)
            self.labels_to_index[class_name] = labels_index
            self.index_to_labels[labels_index] = class_name

            for file_name in os.listdir(class_dir):
                if not file_name.lower().endswith(self.valid_extensions):
                    continue

                path = os.path.join(class_dir, file_name)
                if not os.path.isfile(path):
                    continue
                    
                self.samples.append((path, labels_index))
                self.labels.append(labels_index)
            
    def __len__(self):
        return len(self.samples)

    def num_classes(self):
        return len(self.labels_to_index)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

## CNN Classifer

In [5]:
class CNNClassifier(nn.Module):
    def __init__(self, num_classes):
        
        super().__init__()
        self.num_classes = num_classes
        # Feature extractor
        self.features = nn.Sequential(
            ConvBlock(3, 32),    # 224 → 112
            ConvBlock(32, 64),   # 112 → 56
            ConvBlock(64, 128),  # 56 → 28
            ConvBlock(128, 256), # 28 → 14
        )

        # Adaptive pooling → fixed size
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)   # scalar output
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x  # shape: (batch,num_classes)

## Training Loop

In [6]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    num_epochs=20,
    patience=5,
    verbose=True
):
    model.to(device)

    best_val_acc = 0.0
    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(num_epochs):
        # ================= TRAIN =================
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for batch_idx, (images, targets) in enumerate(train_loader):
            images = images.to(device)
            targets = targets.to(device)

            logits = model(images)
            loss = criterion(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # Compute accuracy
            preds = torch.argmax(logits, dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)

            # -------- Verbose Batch Logging --------
            if verbose and batch_idx % 50 == 0:
                print(f"[Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)}] "
                      f"Loss: {loss.item():.4f}")

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # ================= VALIDATION =================
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device)
                targets = targets.to(device)

                logits = model(images)
                loss = criterion(logits, targets)

                val_loss += loss.item()

                preds = torch.argmax(logits, dim=1)
                correct += (preds == targets).sum().item()
                total += targets.size(0)

        val_loss /= len(val_loader)
        val_acc = correct / total

        # ================= LOG =================
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"\nEpoch [{epoch+1}/{num_epochs}] Summary:")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}\n")

        # ================= EARLY STOPPING (BASED ON ACCURACY) =================
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pth")
        
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    return history

## Utilities

In [7]:
def sample_subset(dataset, num_samples=100, seed=42):
    random.seed(seed)
    indices = list(range(len(dataset)))
    sampled_indices = random.sample(indices, num_samples)
    return sampled_indices

In [8]:
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

shift_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Lambda(lambda img: affine(img, angle=0, translate=(5, 5), scale=1.0, shear=0, fill=0)),
    transforms.ToTensor(),
])

## Dataset and Dataloader Objects

In [28]:
# root_dir= '../plantvillage dataset/segmented'
root_dir= '../plantvillage dataset/color'
train_batch_size = 128
test_batch_size = 64
dataset = PlantVillageDataset(root_dir, transform=base_transform)
dataset_shifted = PlantVillageDataset(root_dir, transform=shift_transform)

train_idx, val_idx, test_idx = stratified_split(dataset)
hundred_test_idx = np.random.choice(test_idx, size = 100, replace= False)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, hundred_test_idx)
test_dataset_shifted = Subset(dataset_shifted, hundred_test_idx)


train_loader = DataLoader(
    train_dataset,
    batch_size=train_batch_size,
    shuffle=True
    # num_workers=2,
    # pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=test_batch_size,
    shuffle=False
    # num_workers=2,
    # pin_memory=True
)

## Driver Code

In [77]:
save_dir = "../Models"
model_num = 2
save_path = os.path.join(save_dir, f"2_6_{model_num}.pth")

# Create directory if it does not exist
os.makedirs(save_dir, exist_ok=True)

# ================= MODEL =================
num_classes = dataset.num_classes()
model = CNNClassifier(num_classes = num_classes)

# ================= LOSS =================
criterion = nn.CrossEntropyLoss()

# ================= OPTIMIZER =================
optimizer = optim.Adam(model.parameters(), lr=5e-4)

# ================= TRAIN =================
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=20,
    patience=3,
    verbose=True
)

# ================= SAVE FINAL MODEL =================
torch.save(model.state_dict(), save_path)
print(f"Model saved to: {save_path}")

[Epoch 1 | Batch 0/297] Loss: 3.6084
[Epoch 1 | Batch 50/297] Loss: 2.5782
[Epoch 1 | Batch 100/297] Loss: 1.9744
[Epoch 1 | Batch 150/297] Loss: 1.5775
[Epoch 1 | Batch 200/297] Loss: 1.2736
[Epoch 1 | Batch 250/297] Loss: 1.0568

Epoch [1/20] Summary:
Train Loss: 1.7925 | Train Acc: 0.5330
Val Loss:   1.0226 | Val Acc:   0.7175

[Epoch 2 | Batch 0/297] Loss: 0.8601
[Epoch 2 | Batch 50/297] Loss: 0.7724
[Epoch 2 | Batch 100/297] Loss: 0.7530
[Epoch 2 | Batch 150/297] Loss: 0.7478
[Epoch 2 | Batch 200/297] Loss: 0.4886
[Epoch 2 | Batch 250/297] Loss: 0.4775

Epoch [2/20] Summary:
Train Loss: 0.7090 | Train Acc: 0.8050
Val Loss:   0.7273 | Val Acc:   0.7886

[Epoch 3 | Batch 0/297] Loss: 0.5858
[Epoch 3 | Batch 50/297] Loss: 0.4987
[Epoch 3 | Batch 100/297] Loss: 0.5696
[Epoch 3 | Batch 150/297] Loss: 0.5008
[Epoch 3 | Batch 200/297] Loss: 0.4177
[Epoch 3 | Batch 250/297] Loss: 0.5839

Epoch [3/20] Summary:
Train Loss: 0.4709 | Train Acc: 0.8660
Val Loss:   0.5326 | Val Acc:   0.8470

[

In [49]:
## Testing
# Train Loss: 0.1507, Acc: 0.9488 | Val Loss: 0.4326, Acc: 0.8770 | Time: 41.33s
# Train Loss: 0.1429 | Train Acc: 0.9580  |  Val Loss:   0.2365 | Val Acc:   0.9303

def count_params(model): 
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [29]:
def evaluate_model(model, dataloader, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    return accuracy

In [45]:
hundred_test_idx = np.random.choice(test_idx, size = 100, replace= False)
# test_dataset  = Subset(dataset, hundred_test_idx)
test_dataset  = Subset(dataset, test_idx)
test_dataset_shifted = Subset(dataset_shifted, test_idx)

num_classes = len(dataset.labels_to_index)
model = CNNClassifier(num_classes=num_classes)

save_dir = "../Models"
model_num = 1
model_path = os.path.join(save_dir, f"2_6_{model_num}.pth")

model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

##########  DATA LOADERS FOR TESTING  ##########
test_loader_base = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

test_loader_shifted = DataLoader(
    test_dataset_shifted,
    batch_size=32,
    shuffle=False
)

In [48]:
########  TEST ACCURACY ORIGINAL   ########
total = 0
correct = 0
for img,label in test_dataset:
    img = torch.unsqueeze(img,0)
    img = img.to(device)
    logits = model(img)
    pred = torch.argmax(logits, dim=1)

    correct += (pred == label).sum().item()
    total +=1

test_accuracy = correct / total
print(f"Test Accuracy (Original): {test_accuracy:.6f}")


########  TEST ACCURACY SHIFTED   ########
total = 0
correct = 0
for img,label in test_dataset_shifted:
    img = torch.unsqueeze(img,0)
    img = img.to(device)
    logits = model(img)
    pred = torch.argmax(logits, dim=1)

    correct += (pred == label).sum().item()
    total +=1

test_accuracy = correct / total
print(f"Test Accuracy (Shifted): {test_accuracy:.6f}")

Test Accuracy (Original): 0.947827
Test Accuracy (Shifted): 0.766757


In [52]:
def load_model(model_class,num_classes, model_path, device):
    model = model_class(num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

def count_params(model): 
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def evaluate_model(model, dataloader, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print("Correct:", correct)
    print("Total:", total)
    
    return accuracy

In [55]:
save_dir = "../Models"
model_num = 1
model_save_path = os.path.join(save_dir, f"2_6_{model_num}.pth")

test_dataset  = Subset(dataset, test_idx)
num_classes = dataset.num_classes()

model = load_model(CNNClassifier, num_classes, model_save_path, device)

cnn_params = count_params(model)
test_loader_base = DataLoader( test_dataset, batch_size=32, shuffle=False)
test_accuracy = evaluate_model(model, test_loader_base, device)

print(f"Test Accuracy : {test_accuracy:.6f}")
print("Number of Params:", cnn_params)
print(f"Accuracy per Parameter : {test_accuracy/cnn_params:.6f}")

Correct: 7721
Total: 8146
Test Accuracy : 0.947827
Number of Params: 408294
Accuracy per Parameter : 0.000002


In [ ]:
## 2.6 Results:
# Model on Color Dataset
# Train Acc: 0.9575
# Val Acc:   0.9489
# Test Accuracy: 0.9300
# Test Accuracy Shifted : 0.8000


## 2.6 Results:
# Model on Segmented Dataset
# Train Acc: 0.9575
# Val Acc:   0.9489
# Test Accuracy: 0.9300
# Test Accuracy Shifted : 0.8000